# 01 - Regressão logística multiclasse

## Objetivo

Prever `AGE_RANGE`, com 16 classes, usando uma divisão fixa de treino/validação e avaliar o desempenho com `Macro F1`.

A preparação dos dados continua centralizada em `src/preparar_dados.py`. Cada experiência reutiliza a preparação comum e cria uma divisão estratificada antes do treino.


## Ambiente e caminhos

Confirma-se o ambiente Python e resolve-se a raiz do projeto de forma portátil para que o notebook funcione independentemente de onde o Jupyter foi iniciado.


In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
print(sys.version)

candidate_roots = [Path.cwd(), Path.cwd().parent]
repo_root = next((p for p in candidate_roots if (p / 'notebooks').exists() and (p / 'src').exists()), Path.cwd())
print(f"repo_root={repo_root}")


3.14.8 | packaged by Anaconda, Inc. | (main, Oct  1 2026, 20:20:32) [Clang 20.1.8 ]
repo_root=/Users/eduardaferreira/daa-kaggle-2026-2027


## 1. Dados e divisão

Vamos carregar os CSV originais, aplicar a preparação comum e fazer uma divisão estratificada para treino/validação, mantendo esta mesma divisão para todas as experiências futuras.


In [10]:
import sys
import importlib

sys.path.insert(0, str(raiz))

for nome in list(sys.modules):
    if nome == "src" or nome.startswith("src."):
        del sys.modules[nome]

importlib.invalidate_caches()

from src.preparar_dados import make_stratified_split, preparar_dados

print("Importação concluída.")

Importação concluída.


In [11]:
from pathlib import Path

import pandas as pd

from src.preparar_dados import make_stratified_split, preparar_dados

repo_root = Path.cwd()
if not (repo_root / 'notebooks').exists() or not (repo_root / 'src').exists():
    repo_root = next((p for p in [Path.cwd(), Path.cwd().parent] if (p / 'notebooks').exists() and (p / 'src').exists()), repo_root)

train_path = repo_root / 'notebooks' / 'dataset_train.csv'
test_path = repo_root / 'notebooks' / 'dataset_test_X.csv'

if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError(f"Ficheiros de dados não encontrados em {repo_root / 'notebooks'}")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
prepared = preparar_dados(train_df, test_df)
X = prepared['X_train']
y = prepared['y_train']

X_train, X_val, y_train, y_val = make_stratified_split(X, y, test_size=0.2, random_state=42)

print(f"treino: {X_train.shape}, {y_train.shape}")
print(f"validacao: {X_val.shape}, {y_val.shape}")
print(f"classes alvo: {sorted(y.unique().tolist())}")
print(f"n_classes={y.nunique()}")


treino: (8769, 236), (8769,)
validacao: (2193, 236), (2193,)
classes alvo: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
n_classes=16


## 2. Modelo inicial

Usamos uma regressão logística multiclasse com regularização L2, `C=1.0` e `solver='saga'`. Na versão do scikit-learn do ambiente, a estratégia multiclasse usa a parametrização multinomial por omissão para este caso.

- `saga` é compatível com classificação multiclasse e dados esparsos.
- `normalize_numeric=True` activa a normalização numérica no pré-processador.
- A pipeline garante a ordem correta: `fit` apenas em treino, `transform` na validação.


In [12]:
from sklearn.linear_model import LogisticRegression

from src.preparar_dados import make_model_pipeline

model = LogisticRegression(
    penalty='l2',
    C=1.0,
    solver='saga',
    max_iter=2000,
    random_state=42,
)

pipe = make_model_pipeline(model, normalize_numeric=True)

print(pipe)
print(f"solver={model.solver}")
print(f"penalty={model.penalty}")
print(f"C={model.C}")


Pipeline(steps=[('preprocessor', DataPreprocessor(normalize_numeric=True)),
                ('model',
                 LogisticRegression(max_iter=2000, penalty='l2',
                                    random_state=42, solver='saga'))])
solver=saga
penalty=l2
C=1.0


## 3. Validação sem leakage

O ajuste da preparação e do modelo só acontece na parcela de treino. Depois, a validação usa `transform` sem refazer o `fit`.


In [13]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

pipe.fit(X_train, y_train)

y_train_pred = pipe.predict(X_train)
y_val_pred = pipe.predict(X_val)

macro_f1_train = f1_score(y_train, y_train_pred, average='macro')
macro_f1_val = f1_score(y_val, y_val_pred, average='macro')

print(f"Macro F1 treino: {macro_f1_train:.4f}")
print(f"Macro F1 validacao: {macro_f1_val:.4f}")
print('--- classificacao por classe ---')
print(classification_report(
    y_val,
    y_val_pred,
    labels=list(range(16)),
    zero_division=0,
    target_names=[str(i) for i in range(16)],
))
print('--- matriz de confusao ---')
print(confusion_matrix(y_val, y_val_pred, labels=list(range(16))))


Macro F1 treino: 0.9093
Macro F1 validacao: 0.3937
--- classificacao por classe ---
              precision    recall  f1-score   support

           0       0.76      0.63      0.69        41
           1       0.30      0.29      0.30        34
           2       0.48      0.49      0.48        80
           3       0.28      0.28      0.28        75
           4       0.23      0.28      0.25        90
           5       0.24      0.26      0.25       129
           6       0.28      0.24      0.26       172
           7       0.37      0.40      0.39       173
           8       0.55      0.51      0.53       181
           9       0.56      0.53      0.54       203
          10       0.52      0.56      0.54       213
          11       0.42      0.44      0.43       208
          12       0.32      0.30      0.31       200
          13       0.28      0.27      0.28       165
          14       0.33      0.35      0.34       130
          15       0.46      0.39      0.43        

## 4. Primeira submissão

A pipeline nova volta a ser ajustada nas 10962 linhas completas do treino e usa a mesma configuração. O teste é previsto na ordem original.

O CSV de submissão é guardado em `submissions/` e não substitui silenciosamente um ficheiro existente.


In [14]:
from pathlib import Path

import pandas as pd

submission_dir = repo_root / 'submissions'
submission_dir.mkdir(exist_ok=True, parents=True)

final_pipe = make_model_pipeline(
    LogisticRegression(
        penalty='l2',
        C=1.0,
        solver='saga',
        max_iter=2000,
        random_state=42,
    ),
    normalize_numeric=True,
)

final_pipe.fit(prepared['X_train'], prepared['y_train'])
y_test_pred = final_pipe.predict(prepared['X_test'])

submission = pd.DataFrame({'AGE_RANGE': y_test_pred.astype(int)})

assert len(submission) == 3655, f"Tamanho inesperado: {len(submission)}"
assert list(submission.columns) == ['AGE_RANGE'], submission.columns
assert submission['AGE_RANGE'].between(0, 15).all(), submission['AGE_RANGE'].describe()

submission_path = submission_dir / 'submission_001_regressao_logistica.csv'
counter = 1
while submission_path.exists():
    counter += 1
    submission_path = submission_dir / f'submission_001_regressao_logistica_{counter}.csv'

submission.to_csv(submission_path, index=False)
print(f"Submissao guardada em: {submission_path}")
print(f"linhas={len(submission)}")
print(submission.head())


Submissao guardada em: /Users/eduardaferreira/daa-kaggle-2026-2027/submissions/submission_001_regressao_logistica_2.csv
linhas=3655
   AGE_RANGE
0          4
1          3
2         10
3          6
4          9


## 5. Registo após submissão

Completar após a submissão no Kaggle.

- Data:  8/11  
- Macro F1 treino: 0.9093
- Macro F1 validação: 0.3937
- Pontuação pública no Kaggle:  0.43
- Conclusões: Diferença elevada entre treino e validação sugere overfitting
- Melhorias a experimentar: Reduzir C para aumentar a regularização; testar class_weight="balanced"; verificar convergência e avaliar com validação cruzada
Investigar a separação exata das previsões aos 55 anos e verificar se está associada a perguntas condicionadas pela idade.
